# Finding topics with BERTopic

In session 7 we sorted paragraphs into categories that **we chose in advance**: *social class*, *nature*, *love*. But what if we don't know which themes are in a corpus? Here we ask a different question:

> **Which groups of paragraphs talk about similar things?**

**BERTopic** is a method for **topic modelling**: it finds groups of similar texts by itself, and describes each group with its most characteristic words. We never tell it which topics to look for. This is called **unsupervised** learning.

In this notebook you will:
1. understand the five steps of BERTopic, and how they relate to what we have already seen;
2. run it on `data/ALL_paragraphs`, the 150 paragraphs from the three novels;
3. read the results critically, and do the exercises at the end.

## What BERTopic does, step by step

| Step | What happens | Tool | Have we seen it? |
|---|---|---|---|
| 1. **Embed** | Each paragraph becomes a list of numbers that represents its meaning | A transformer model | Transformers and contextual embeddings: session 7 |
| 2. **Reduce** | These lists of numbers are compressed, so that they are easier to group | UMAP | New |
| 3. **Cluster** | Paragraphs that are close to each other form a group: these are the topics | HDBSCAN | New |
| 4. **Count words** | We count the words of each topic | Bag of Words | Session 5 |
| 5. **Find characteristic words** | We keep the words that characterise each topic | c-TF-IDF, a variant of TF-IDF | Session 5 |

Note the division of labour:
- **Steps 1-3 use meaning** to create the groups.
- **Steps 4-5 use word counts** to describe the groups, in a way that we can read.

### How does it compare with the classification of session 7?

| | Zero-shot (BART-MNLI) | Few-shot LLM | BERTopic |
|---|---|---|---|
| Who decides the categories? | We do | We do | The method finds groups, and we interpret them |
| Do we need to describe the categories? | Yes, with hypotheses | Yes, with definitions and examples | No |
| What do we get for each paragraph? | A category | A category and an explanation | A topic number, or "no topic" |
| What is it good for? | Testing known categories | Testing known categories | Exploring a corpus |

The name comes from **BERT**, a famous transformer model. BERTopic can use various models for step 1, and the default ones are related to BERT.

## Installation

This cell installs BERTopic and everything it needs. You only need to run it once in an environment; it can take a few minutes.

`--extra-index-url` asks for the version of PyTorch made for CPUs, which is much smaller. BERTopic runs fine without a GPU on a corpus of this size.

In [ ]:
# Installs BERTopic. It brings along sentence-transformers (step 1), UMAP (step 2), HDBSCAN (step 3),
# scikit-learn (steps 4 and 5) and pandas (tables).
# -q means "quiet": pip prints fewer messages.
%pip install -q bertopic --extra-index-url https://download.pytorch.org/whl/cpu

## Load the paragraphs

As in the homework of session 7: one row per paragraph, with the name of the file and its text. The first four letters of the file name tell us the novel, which will be useful later.

In [ ]:
from pathlib import Path  # Provides tools for locating files and folders.
import pandas as pd       # Provides DataFrames for tables.

pd.set_option("display.max_colwidth", 120)  # Shows longer texts in table cells.

DATA_DIR = Path("../data/ALL_paragraphs")   # The folder with the paragraphs.

# One row per file: its name and its text.
# sorted(...) gives the files always in the same order; strip() removes spaces and line breaks at the edges of the text.
df = pd.DataFrame([{"file": path.name, "text": path.read_text(encoding="utf-8").strip()}
                   for path in sorted(DATA_DIR.glob("*.txt"))])

# "CATH-ALL-23_01.txt" -> "CATH": the first four letters of the file name identify the novel.
df["novel"] = df["file"].str[:4]

# BERTopic wants a simple list of texts, so we prepare one.
paragraphs = df["text"].tolist()

print(len(df), "paragraphs")
df["novel"].value_counts()  # How many paragraphs per novel? (CATH = Cather, FITZ = Fitzgerald, HEMI = Hemingway)

## Step 1. Embed: from text to meaning

An **embedding** is a list of numbers that represents a text. It is still a vector, like in session 5, but there is an important difference:

| | Bag of Words (session 5) | Embedding |
|---|---|---|
| What does each number mean? | The count of one specific word | Nothing we can name: the model learned it |
| How long is the list? | As long as the vocabulary, with many zeros | Short here: 384 numbers, none of them empty |
| Do *gossip* and *rumours* look similar? | No | Yes |

As we saw in session 7, a transformer gives a different vector to a word depending on its context (**contextual embeddings**). Here the model averages the vectors of the words of a paragraph, to obtain one vector for the whole paragraph.

We use the model [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), BERTopic's default for English. It is small (about 90 MB), and it is downloaded the first time you run the next cell.

In [ ]:
import os  # Gives access to the settings of the operating system.

os.environ["HF_HUB_DISABLE_XET"] = "1"          # Only needed in mybinder, Google Colab, or similar environments.
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Hides a harmless warning.

from sentence_transformers import SentenceTransformer  # The library that runs the embedding model.

# Loads the model (and downloads it, the first time). You may see some messages in red about loading weights
# or about a missing HF_TOKEN: you can ignore them.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Turns every paragraph into one vector. This takes a moment on a CPU.
embeddings = embedding_model.encode(paragraphs, show_progress_bar=False)

# The result is a table of numbers: one row per paragraph, one column per dimension.
print("Rows (paragraphs), columns (dimensions):", embeddings.shape)
print("The first 5 numbers of the first paragraph:", embeddings[0][:5].round(3))

**Attention: long paragraphs are cut.** This model only reads a limited number of tokens, and it ignores the rest of the text. Remember from session 7 that a transformer's token is not always a whole word. Let's check how many of our paragraphs are affected.

In [ ]:
limit = embedding_model.max_seq_length  # The maximum number of tokens that the model reads.

# Counts the tokens of each paragraph, using the model's own tokenizer.
# You may see a warning that a text is too long for the model: that is exactly what we are checking here.
# ["input_ids"] is the list of tokens, as numbers: we only need its length.
token_counts = [len(embedding_model.tokenizer(text)["input_ids"]) for text in paragraphs]

# Counts how many paragraphs are longer than the limit.
too_long = sum(count > limit for count in token_counts)

print(f"The model reads at most {limit} tokens. {too_long} of {len(paragraphs)} paragraphs are longer than that.")

### Do embeddings really capture meaning?

To compare two vectors we use the **cosine similarity**: 1 means they point in the same direction, values near 0 mean they have little in common. Let's test it on the example that Bag of Words could not handle in session 5.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity  # Measures how similar two vectors are.

sentences = ["There was a lot of gossip about him in the town.",
             "There were many rumours about him in the town.",
             "The river was frozen in the town."]

# One vector for each of the three sentences.
vectors = embedding_model.encode(sentences)

# Compares every sentence with every other sentence, and shows the result as a table.
similarity = pd.DataFrame(cosine_similarity(vectors),
                          index=["gossip", "rumours", "river"],
                          columns=["gossip", "rumours", "river"]).round(2)
similarity

The same comparison works on our paragraphs. Which paragraphs does the model consider closest to a given one? Read them: do you agree?

In [ ]:
FILE = "CATH-ALL-23_03.txt"  # The paragraph to start from. CHANGE HERE to try another one.

# Finds the row number of this paragraph in the table.
position = df.index[df["file"] == FILE][0]

# Compares this paragraph with all the paragraphs. [0] takes the only row of the result.
scores = cosine_similarity(embeddings[[position]], embeddings)[0]

# Makes a copy of the table with the scores, sorts it from the most to the least similar and keeps the first 4 rows.
# The first row is the paragraph itself, with a score of 1.
df.assign(similarity=scores.round(2)).sort_values("similarity", ascending=False).head(4)[["file", "similarity", "text"]]

## Step 2. Reduce: UMAP

Clustering methods work poorly with many dimensions, so **UMAP** compresses our 384 numbers into a few, trying to keep together the paragraphs that were close.

It's like a map: a globe has three dimensions and a map only two, so something gets distorted, but neighbouring cities stay neighbours.


In [ ]:
from umap import UMAP  # The tool for step 2.

umap_model = UMAP(
    n_neighbors=10,     # How many neighbours each paragraph looks at. Small: local detail. Large: the big picture.
    n_components=5,     # How many dimensions to keep: from 384 down to 5.
    min_dist=0.0,       # Allows paragraphs to be packed tightly together, which helps to form groups in step 3.
    metric="cosine",    # Measures closeness with the cosine similarity, as above.
    random_state=42,    # The seed: fixes the randomness. Any number works, but results change with it. UMAP includes some randomness, so we fix its **seed** (`random_state`) to get the same result when we run the notebook again.
)

## Step 3. Cluster: HDBSCAN

**HDBSCAN** looks for dense regions, where many paragraphs are close together. Each dense region is a **cluster**, and each cluster will be a topic.

Two things to notice:
- **We don't say how many topics we want**: the number depends on the data and on the settings.
- **A paragraph can stay out of all topics.** These **outliers** get the number `-1`. This is a difference from the zero-shot classification of session 7, which had to choose a category even if none fitted.

In [ ]:
from hdbscan import HDBSCAN  # The tool for step 3.

hdbscan_model = HDBSCAN(
    min_cluster_size=5,    # A topic must have at least 5 paragraphs. This is the most important setting here.
    metric="euclidean",    # After UMAP, closeness is measured as ordinary distance, like on a map.
    prediction_data=True,  # Keeps some extra information that BERTopic needs.
)

## Steps 4 and 5. Describe: Bag of Words and c-TF-IDF

Now we have groups, but only as numbers. To know what they are about, we need words.

**Step 4** is the Bag of Words from session 5, with `CountVectorizer`. We remove the stop words (session 4), otherwise *the* and *and* would describe every topic.

**Step 5** is **c-TF-IDF** (*class-based TF-IDF*). In session 5, TF-IDF gave a high score to words that are frequent in one *document* and rare in the others. Here we use the same idea, with one change: **all the paragraphs of a topic are joined together and treated as one big document**. So a high score means that the word is frequent in this *topic* and rare in the others.

BERTopic takes care of step 5 by itself, so we only need to prepare step 4.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer  # The tool for step 4, already used in session 5.

# stop_words="english" removes common English words, only from the description of the topics.
vectorizer_model = CountVectorizer(stop_words="english")

## Put it all together

We give BERTopic the tools of each step, and it runs them in the right order. We already calculated the embeddings, so we pass them directly and save some time.

In [ ]:
from bertopic import BERTopic  # The library that connects all the steps.

topic_model = BERTopic(
    umap_model=umap_model,              # Step 2.
    hdbscan_model=hdbscan_model,        # Step 3.
    vectorizer_model=vectorizer_model,  # Step 4. (Step 5, c-TF-IDF, is the default.)
)

# Runs the pipeline. It returns one topic number per paragraph (and a probability that we don't use here: "_").
topics, _ = topic_model.fit_transform(paragraphs, embeddings)

# Saves the topic number next to each paragraph.
df["topic"] = topics

# One row per topic: its number, its size and an automatic name made with its top words.
# Topic -1 is not a real topic: it collects the outliers.
topic_model.get_topic_info()[["Topic", "Count", "Name"]]

## Read the results

**The topics have no real names.** A name like `0_waiter_came_table_car` is only a list of top words. Giving a topic a meaningful name is **your interpretation**, and you need to read the paragraphs to do it.

First, the top words of one topic, with their c-TF-IDF scores.

In [ ]:
TOPIC = 1  # The topic to look at. CHANGE HERE to see another one.

# get_topic returns a list of pairs (word, score), from the highest score down.
pd.DataFrame(topic_model.get_topic(TOPIC), columns=["word", "c-TF-IDF"]).round(3)

Then the paragraphs. BERTopic selects a few **representative documents** for each topic.

In [ ]:
# Selects the same topic as above and prints its representative paragraphs.
for text in topic_model.get_representative_docs(TOPIC):
    print(text[:500], "...\n")  # Prints only the first 500 characters of each paragraph.

Finally, all the paragraphs of a topic, and the outliers. Why did the outliers stay out? Are they really about something else?

In [ ]:
# All the paragraphs of the selected topic.
df[df["topic"] == TOPIC][["file", "text"]]

In [ ]:
# The paragraphs that were not assigned to any topic.
df[df["topic"] == -1][["file", "text"]]

## Topics or novels?

Do the topics cross the novels, or do they simply separate them? A **crosstab** counts the paragraphs for each combination, like the confusion matrix of session 7.

In [ ]:
# Rows = topics, columns = novels. Each cell counts the paragraphs of that topic in that novel.
pd.crosstab(df["topic"], df["novel"])

This is an important result to think about. In our run, several topics come almost entirely from one novel, and some of their top words are character names. So the method may be finding **novels and characters** as much as themes.

This is a good reminder: the method finds what makes texts similar, which is not necessarily what we were looking for. Exercise 3 explores this.

## Compare with the categories of session 7

Do the topics match *social class*, *nature* and *love*? Here we use `gt_fake.csv` from the homework of session 7. Remember that those labels **were made by an LLM**, and that some paragraphs have no label, so this only compares two methods: it does not tell us who is right.

In [ ]:
# Reads the labels of session 7: one row per file, with its category (some are empty).
labels = pd.read_csv("../session07/homework/gt_fake.csv")

# Adds the category to our table, matching the rows by file name. how="left" keeps all our paragraphs.
merged = df.merge(labels, on="file", how="left")

# Rows = topics, columns = categories. Paragraphs without a category are shown as "(no label)".
pd.crosstab(merged["topic"], merged["category"].fillna("(no label)"))

## Things to keep in mind

- **The topics are not facts.** They depend on the settings, on the model and on the seed. Exercise 2 shows how much.
- **Our corpus is small.** BERTopic usually works with thousands of documents. With 150 paragraphs the results are less stable, so treat them as a starting point for reading.
- **There is no ground truth here,** so there is no accuracy either. Check the results by reading the paragraphs. If you have your own annotations, you can compare them with a crosstab.
- **The unit of analysis matters.** Paragraphs can contain several themes, but BERTopic assigns only one topic to each.
- **Stop words only affect the description.** They are used in steps 4-5, so they change the top words, but not the groups.
- **Pre-processing matters.** BERTopic removes punctuation before counting the words, so *he's* becomes `hes`. This is not in the list of stop words, which is why it can appear among the top words.
- **Language.** This embedding model is for English. For other languages, try `paraphrase-multilingual-MiniLM-L12-v2` (change only the name of the model).

Want to know more? See the [BERTopic documentation](https://maartengr.github.io/BERTopic/) and the original article: Maarten Grootendorst, "BERTopic: Neural Topic Modeling with a Class-Based TF-IDF Procedure", 2022, https://arxiv.org/abs/2203.05794.

# Exercises

Create a new Markdown cell after each exercise to write your answers.

## Exercise 1. Name the topics

For each topic, change `TOPIC` in the cells above and read the top words and the representative paragraphs.

1. Give each topic a name in your own words. Is it about a theme, a place, a character, a style, or a novel?
2. Which topics are the clearest? Which ones can't you name?
3. Read five outliers. Should they belong to a topic?

## Exercise 2. How stable are the topics?

The function below repeats the whole pipeline with the settings that you choose.

1. Change `min_cluster_size` to 3, then to 10. How many topics and outliers do you get?
2. Put `min_cluster_size` back to 5 and change `seed` to 1, then to 2. Do the topics stay the same? What does this mean for how you can use them in an analysis?

In [ ]:
def find_topics(texts, vectors, min_cluster_size=5, n_neighbors=10, seed=42):
    # Runs the pipeline with new settings, on the given texts and their embeddings.
    # It returns the model and the topic number of each text.
    model = BERTopic(
        umap_model=UMAP(n_neighbors=n_neighbors, n_components=5, min_dist=0.0, metric="cosine", random_state=seed),
        hdbscan_model=HDBSCAN(min_cluster_size=min_cluster_size, metric="euclidean", prediction_data=True),
        vectorizer_model=CountVectorizer(stop_words="english"),
    )
    found_topics, _ = model.fit_transform(texts, vectors)
    return model, found_topics

# CHANGE HERE: try other values for min_cluster_size and seed.
new_model, new_topics = find_topics(paragraphs, embeddings, min_cluster_size=5, seed=42)

# Shows the topics found with these settings.
new_model.get_topic_info()[["Topic", "Count", "Name"]]

## Exercise 3. Topics or character names?

Are the topics shaped by character names? To find out, we remove the names **from the text, before the embeddings**, with a regular expression (session 1). Adding them to the stop words would not be enough: stop words only change the description of the topics, not the groups.

1. Run the cell below. Compare the crosstab with the one above: do the topics now mix the novels more?
2. Add to `NAMES` the other names that you find in the top words, and run the cell again.
3. Do you get more thematic topics? What do you lose by removing the names?

In [ ]:
import re  # Regular expressions (session 1).

# The names to remove. CHANGE HERE: add the other names that you find in the top words.
NAMES = ["Forrester", "Niel", "Daisy", "Gatsby", "Tom", "Brett", "Romero", "Cohn", "Jake", "Bill", "Mike"]

# Builds a regular expression like \b(Forrester|Niel|Daisy)('s|’s)?\b, which also matches the possessive ("Daisy's").
# \b means "the edge of a word", so that "Tom" does not match "Tomorrow".
pattern = r"\b(" + "|".join(NAMES) + r")('s|’s)?\b"

# Replaces each name with the neutral word "someone", in every paragraph.
paragraphs_no_names = [re.sub(pattern, "someone", text, flags=re.IGNORECASE) for text in paragraphs]

# The text has changed, so we must calculate the embeddings again.
embeddings_no_names = embedding_model.encode(paragraphs_no_names, show_progress_bar=False)

# Runs the pipeline on the new texts, with the same settings as before.
model_no_names, topics_no_names = find_topics(paragraphs_no_names, embeddings_no_names)

# Shows the new topics...
print(model_no_names.get_topic_info()[["Topic", "Count", "Name"]])

# ... and how they are distributed across the novels.
pd.crosstab(pd.Series(topics_no_names, name="topic"), df["novel"])